# coordinates/teldef

迁移自 / Migrated from `test/teldef_test.ipynb`.

状态 / Status: **needs_review**. 需要 test/data 中的 teldef 校准文件 / Requires the teldef calibration fixture

历史输出保留，不代表迁移后重新验证；原报错数量 / Preserved historical errors: 0.
Historical outputs are retained and are not evidence of a new successful run.

输入路径在下一单元集中配置；原始数据留在原位置。
Input roots are configured below; original data remain in place.

脚本试运行完成，但坐标往返不一致；算法需单独复核。
Script smoke completed but round-trip coordinates disagree; separate algorithm review is required.


In [ ]:
# 从仓库定位输入，允许通过环境变量覆盖；不移动原始数据。
# Locate repository inputs with environment overrides; original data stay in place.
from pathlib import Path
from datetime import datetime, timezone
import os
_anchor = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
REPO_ROOT = next((p for p in (_anchor, *_anchor.parents)
                  if (p/'packages/jinwu').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('请从 JinWu 仓库内运行 / Run from within the JinWu checkout')
RESEARCH_ROOT = Path(os.environ.get('JINWU_EXAMPLE_RESEARCH_ROOT', REPO_ROOT.parent))
TEST_DATA_ROOT = Path(os.environ.get('JINWU_EXAMPLE_TEST_ROOT', REPO_ROOT/'test'))
DOWNLOAD_ROOT = Path(os.environ.get('JINWU_EXAMPLE_DOWNLOAD_ROOT', Path.home()/'下载'))
# 图件写入独立运行目录，不覆盖研究目录中的历史图。
# Write figures into a unique run directory, preserving historical research plots.
OUTPUT_DIR = Path(os.environ.get('JINWU_EXAMPLE_OUTPUT_ROOT', REPO_ROOT/'examples/_outputs')) / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)


In [2]:
from pathlib import Path
from jinwu.ftools import teldef
import numpy as np
p = Path(str(TEST_DATA_ROOT / 'data/swx20230701v001.teldef')).resolve()
print('teldef path:', p)
T = teldef.Teldef.from_file(str(p))
print('has align:', hasattr(T,'align'))
if hasattr(T,'align'):
    print('ALIGNM:')
    print(T.align)
    print('focal_length:', getattr(T,'focal_length',None))
    print('det_xscl, det_yscl:', getattr(T,'det_xscl',None), getattr(T,'det_yscl',None))
    print('optaxis:', getattr(T,'optaxis',None))
# example pointing (RA_PNT, DEC_PNT) -- replace with real pointing when available
ra_pnt = 180.0
dec_pnt = 45.0
# sample sky coordinate near boresight
ra = 180.001
dec = 45.0
if hasattr(T,'align'):
    xpix, ypix = T.sky_to_det_with_pointing(ra, dec, ra_pnt, dec_pnt)
    print('sky->det (with pointing) =>', xpix, ypix)
    ra2, dec2 = T.det_to_sky_with_pointing(xpix, ypix, ra_pnt, dec_pnt)
    print('det->sky =>', ra2, dec2)
else:
    print('teldef has no ALIGNM, use simple sky_to_det/det_to_sky')
    if T.crval is not None and T.crpix is not None:
        print(T.sky_to_det(ra,dec))
        print(T.det_to_sky(*T.sky_to_det(ra,dec)))

teldef path: /home/xinxiang/research/jinwu/test/data/swx20230701v001.teldef
has align: True
ALIGNM:
[[-1.38145185e-04 -2.90887332e-03  9.99995760e-01]
 [ 9.60392834e-06  9.99995769e-01  2.90887468e-03]
 [-9.99999990e-01  1.00057346e-05 -1.38116664e-04]]
focal_length: 3500.0
det_xscl, det_yscl: 0.04 0.04
optaxis: (300.0, 300.0)
sky->det (with pointing) => -87199.99931021415 300.8723672478508
det->sky => 234.6492374636717 29.90758495085327
